### Scenario - 1
**Managed Catalog** <br>
**Managed Schema** <br>
**Managed Table** <br>


## Managed Catalog

In [0]:
%sql

--DROP CATALOG managed_catalog CASCADE;

--If the catalog contains schemas/tables, CASCADE tells Unity Catalog to drop the catalog and its objects underneath it.

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS managed_catalog

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS managed_catalog.managed_schema

In [0]:
%sql
DROP TABLE managed_catalog.managed_schema.managed_table

In [0]:
%sql
SELECT metastore_name, storage_root_credential_id, storage_root_credential_name
FROM system.information_schema.metastores;

-- "Root storage credential for metastore does not exist, while creating managed_table 
-- To resolve: created a 'External Location', which being used as ManagedLocation by metastore.
-- https://community.databricks.com/t5/data-engineering/error-creating-catalog-in-unity-catalog-external-location-does/td-p/128965


metastore_name,storage_root_credential_id,storage_root_credential_name
metastore_neeraj-centralindia,null,null


In [0]:
%sql
-- to use 3 level namespace for creating tables in Unity Catalog

CREATE TABLE IF NOT EXISTS managed_catalog.managed_schema.managed_table(
    id INT,
    name STRING
)
USING DELTA

### Scenario - 2
**External Catalog** <br>
**Managed Schema** <br>
**Managed Table** <br>


In [0]:
%sql
CREATE CATALOG external_catalog
MANAGED LOCATION 'abfss://mycontainer@adlspracticeproject02.dfs.core.windows.net/ext_catalog_data'


In [0]:
%sql
CREATE SCHEMA external_catalog.managed_schema


In [0]:
%sql
CREATE TABLE external_catalog.managed_schema.managed_table2(
    id INT,
    name STRING
)
USING DELTA


### Scenario - 3
**External Catalog** <br>
**External Schema** <br>
**Managed Table** <br>


In [0]:
%sql
CREATE SCHEMA external_catalog.external_schema
MANAGED LOCATION 'abfss://mycontainer@adlspracticeproject02.dfs.core.windows.net/external_schema_data'

In [0]:
%sql
CREATE TABLE external_catalog.external_schema.managed_table3(
    id INT,
    name STRING
)
USING DELTA

### Scenario - 4

**External Table** <br>


In [0]:
%sql
CREATE TABLE managed_catalog.managed_schema.external_table4(
    id INT,
    name STRING
)
USING DELTA
LOCATION 'abfss://mycontainer@adlspracticeproject02.dfs.core.windows.net/external_table_data'

# Drop Managed Table

In [0]:
%sql

Drop table managed_catalog.managed_schema.managed_table

# UnDrop Managed Table

In [0]:
%sql
Undrop table managed_catalog.managed_schema.managed_table

In [0]:
%sql
SHOW TBLPROPERTIES managed_catalog.managed_schema.managed_table;

--as Retains data for 7days by default, before droping of the table.

key,value
delta.enableDeletionVectors,true
delta.enableRowTracking,true
delta.feature.appendOnly,supported
delta.feature.deletionVectors,supported
delta.feature.domainMetadata,supported
delta.feature.invariants,supported
delta.feature.rowTracking,supported
delta.minReaderVersion,3
delta.minWriterVersion,7
delta.parquet.compression.codec,zstd


In [0]:
%sql
DESCRIBE EXTENDED managed_catalog.managed_schema.managed_table

col_name,data_type,comment
id,int,null
name,string,null
,,
# Detailed Table Information,,
Catalog,managed_catalog,
Database,managed_schema,
Table,managed_table,
Created Time,Wed Sep 09 19:31:11 UTC 2026,
Last Access,UNKNOWN,
Created By,Spark,


## Note:
    Your path being inside the metastore's managed storage root doesn't make **table1** managed. 
    However, in practice, you generally shouldn't manually create external tables inside a UC-managed storage location. 
    Keep your managed-table storage and external-table locations conceptually separate

                        CREATE TABLE
                         │
              ┌──────────┴──────────┐
              │                     │
        LOCATION given?        No LOCATION
              │                     │
             YES                    NO
              ↓                     ↓
       EXTERNAL TABLE         MANAGED TABLE


### In your exact Example of below:

Unity Catalog
│
└── managed_catalog
      │
      └── managed_schema
            │
            └── table1       ← EXTERNAL TABLE
                               │
                               ↓
                    abfss://.../check
                               │
                               ↓
                    ADLS container
                    metastoreroot
    


The table metadata is registered in Unity Catalog, while the Delta data lives at:

abfss://metastoreroot@adlspracticeproject02.dfs.core.windows.net/check

In [0]:
%sql
CREATE TABLE managed_catalog.managed_schema.table1(
    id INT,
    name STRING
)
USING DELTA
LOCATION 'abfss://metastoreroot@adlspracticeproject02.dfs.core.windows.net/check'

-- trying to create using Location of managed table, however concept: if specified 'LOCATION' -> is external table. (doesn't depend on which location is specified)